In [ ]:
import polars as pl
import polars.selectors as cs


from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset

config = Config(_env_file="../.env")

data = Dataset.load(config)

Questions shown to new participants in wave 1, then repeating in waves 2,3,4,5

In [ ]:
tail_selection_df = pl.DataFrame(
    {
        "tail_w2": [bool((i // 8) % 2) for i in range(16)],
        "tail_w3": [bool((i // 4) % 2) for i in range(16)],
        "tail_w4": [bool((i // 2) % 2) for i in range(16)],
        "tail_w5": [bool((i // 1) % 2) for i in range(16)],
    }
).lazy()

In [ ]:
heads = (
    # data.codebook.filter(
    #     # pl.col("codebook_name").str.starts_with("dem") |
    #     pl.col("codebook_name").str.starts_with("ew")
    #     | pl.col("codebook_name").str.starts_with("cc")
    #     | pl.col("codebook_name").str.starts_with("cvcc")
    #     |
    #     # pl.col("codebook_name").str.starts_with("soc") |
    #     pl.col("codebook_name").str.starts_with("pol"),
    #     ~pl.col("codebook_name").str.starts_with("pol_worry"),
    #     ~pl.col("codebook_name").str.starts_with("pol_trust"),
    # )
    data.codebook.group_by("item_name", maintain_order=True)
    .agg(pl.col(r"^w\d_(new|rep)$").any())
    .with_row_index("cb_index")
    # .filter(pl.col("item_name").is_in(["dem_stcount_1", "dem_stcount_2"]))
    # .with_columns()
    # .filter()
    .unpivot(
        cs.matches(r"^w\d_new$"),
        index=~cs.matches(r"^w\d_new$"),
        variable_name="wave",
        value_name="is_head",
    )
    .filter("is_head")
    .drop("is_head")
    .with_columns(pl.col("wave").str.extract(r"^w(\d)_new$", 1).cast(int))
    .rename({"wave": "head"})
    .sort(by=("cb_index", "head"))
)

non_empty_tails = (
    heads.join(tail_selection_df, how="cross")
    # Filter: Require that wX in tail ==> wX is present
    .filter(
        ~pl.col("tail_w2") | pl.col("w2_rep"),
        ~pl.col("tail_w3") | pl.col("w3_rep"),
        ~pl.col("tail_w4") | pl.col("w4_rep"),
        ~pl.col("tail_w5") | pl.col("w5_rep"),
    )
    # Gather "true" values into list of tail wave numbers
    .with_row_index("wave_combination_index")
    .unpivot(
        cs.matches(r"^tail_w\d$"),
        index=["cb_index", "wave_combination_index", "item_name", "head"],
        variable_name="tail",
        value_name="included",
    )
    .filter("included")
    .drop("included")
    .with_columns(pl.col("tail").str.extract(r"^tail_w(\d)$", 1).cast(int))
    .group_by(
        "cb_index", "wave_combination_index", "item_name", "head", maintain_order=True
    )
    .agg(pl.col("tail"))
    .drop("wave_combination_index")
    # Require that all tail waves are strictly after the head
    .filter(pl.col("tail").list.min() > pl.col("head"))
    .sort(by=("cb_index", "head", "tail"))
)

empty_tails = (
    heads.select("cb_index", "item_name", "head", pl.lit([]).alias("tail"))
    .unique()
    .sort(by=("cb_index", "head"))
)


# Combine head + tails into single list of waves
heads_and_tails = (
    pl.concat(
        (
            non_empty_tails,
            empty_tails,
        )
    )
    .sort(by=("cb_index", "head", "tail"))
    .with_columns(
        pl.col("head").repeat_by(1).list.concat("tail").list.sort().alias("waves")
    )
    .drop("head", "tail")
)

# Collect head and tail into single list
heads_and_tails.collect()

In [ ]:
(
    heads_and_tails.filter(pl.col("waves").list.len() == 5)
    .group_by("waves")
    .agg(pl.len().alias("n_questions"))
    .sort(by="n_questions")
).collect()

In [ ]:
heads_and_tails.filter(pl.col("waves") == [1, 2, 3, 4]).collect().select(
    "item_name"
).to_series().to_list()

In [ ]:
(
    heads_and_tails.join(heads_and_tails, how="left", on="cb_index").filter(
        pl.col("waves") != pl.col("waves_right"),
        pl.col("waves").list.set_union("waves_right") == pl.col("waves_right"),
    )
).collect()

In [ ]:
empty_tails.filter(cb_index=0, head=1).collect()

In [ ]:
non_empty_tails.filter(cb_index=0, head=1).collect()

In [ ]:
data.codebook.collect()